# Transfer Learning for SteelBlast Image Classification

Using pre-trained neural networks (VGG16, InceptionV3, ResNet50) to classify steel surfaces as "good" or "not-good" for shot-blasting quality assessment.

## 1. Import Required Libraries

In [ ]:
from __future__ import annotations

import os
import json
import sys
import tempfile
from pathlib import Path
from dataclasses import dataclass

# Set matplotlib backend before importing
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

# Reproducibility seed
RANDOM_SEED = 42
os.environ.setdefault('PYTHONHASHSEED', str(RANDOM_SEED))
np.random.seed(RANDOM_SEED)
import random
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import VGG16, InceptionV3, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau




## 2. Pipeline Configuration

## Model configuration

> The values below balance model quality, training stability, and runtime cost for this SteelBlast binary classification task.

- **`image_size = (224, 224)`**
  - Chosen as the default because **ResNet50** and **VGG16** natively use 224x224 inputs.
  - This keeps GPU memory usage and training time lower than 299x299.
  - For **InceptionV3**, switching to **299x299** is appropriate because its architecture was designed for that resolution and usually performs better with it.

- **`batch_size = 32`**
  - A common, stable starting point that gives reasonably smooth gradient estimates without excessive memory pressure.
  - Large enough for efficient throughput on most GPUs/Apple Silicon setups, while still likely to fit in memory.
  - If out-of-memory occurs, reduce to 16 or 8.

- **`epochs = 20`**
  - Provides enough iterations for transfer learning convergence on a moderate dataset.
  - Combined with **EarlyStopping**, training can stop earlier when validation loss stops improving, so 20 acts as a safe upper bound rather than a fixed cost.



In [ ]:
detected_in_colab = "google.colab" in sys.modules
print(f"Running in Colab: {detected_in_colab}")
print(f"TensorFlow version: {tf.__version__}")

@dataclass(frozen=True)
class AppConfig:
    dataset_dir: Path = Path("doi-10.34894-ekznn0(1)/SteelBlastQC")
    # Model choice: 'VGG16', 'InceptionV3', or 'ResNet50'
    model_choice: str = 'ResNet50'
    image_size: tuple = (224, 224)  # VGG16 and ResNet50 use 224x224, InceptionV3 uses 299x299
    batch_size: int = 32
    epochs: int = 20
    learning_rate: float = 0.001
    validation_split: float = 0.2
    quick_limit: int | None = None
    in_colab: bool = False
    local_env: bool = True    # User-controlled flag for local-only setup
    enable_gpu: bool = False  # User-controlled flag
    use_gpu: bool = False     # Effective value after runtime checks

LABELS = {"good": 0, "not-good": 1}
CLASS_NAMES = ["good", "not-good"]

app_config = AppConfig(in_colab=detected_in_colab, local_env=not detected_in_colab)

# Adjust image size based on model choice
if app_config.model_choice == 'InceptionV3':
    app_config = AppConfig(
        image_size=(299, 299),
        model_choice='InceptionV3',
        in_colab=app_config.in_colab,
        local_env=app_config.local_env,
        enable_gpu=app_config.enable_gpu,
    )

# Execute local-only environment variables only when local_env is enabled.
if app_config.local_env:
    os.environ.setdefault("TF_GPU_ALLOCATOR", "metal")
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    print("Applied local environment variables (TF_GPU_ALLOCATOR, TF_CPP_MIN_LOG_LEVEL)")
else:
    print("Skipped local environment variables because local_env=False")

# Detect and optionally enable GPU based on config flag
detected_gpus = tf.config.list_physical_devices('GPU')
gpu_available = len(detected_gpus) > 0

if gpu_available and not app_config.enable_gpu:
    # Disable GPU visibility so TensorFlow runs on CPU when the flag is off.
    try:
        tf.config.set_visible_devices([], 'GPU')
        detected_gpus = []
        gpu_available = False
        print("GPU disabled by AppConfig.enable_gpu=False")
    except RuntimeError as e:
        print("Could not disable GPU after initialization:", e)

use_gpu = app_config.enable_gpu and gpu_available

if use_gpu:
    try:
        for gpu in detected_gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        tf.config.optimizer.set_jit(False)
        print("Enabled memory growth for GPU; XLA JIT disabled")
    except RuntimeError as e:
        print("Could not set GPU memory growth:", e)

if use_gpu:
    policy = tf.keras.mixed_precision.Policy("mixed_float16")
    tf.keras.mixed_precision.set_global_policy(policy)
    print(f"Enabled mixed precision policy: {policy.name}")
else:
    print("Mixed precision disabled because GPU use is off or no GPU was found")

app_config = AppConfig(
    dataset_dir=app_config.dataset_dir,
    model_choice=app_config.model_choice,
    image_size=app_config.image_size,
    batch_size=app_config.batch_size,
    epochs=app_config.epochs,
    learning_rate=app_config.learning_rate,
    validation_split=app_config.validation_split,
    quick_limit=app_config.quick_limit,
    in_colab=app_config.in_colab,
    local_env=app_config.local_env,
    enable_gpu=app_config.enable_gpu,
    use_gpu=use_gpu,
)

print("JIT enabled after config:", tf.config.optimizer.get_jit())
print(f"Model: {app_config.model_choice}")
print(f"Image size: {app_config.image_size}")
print(f"Dataset directory: {app_config.dataset_dir}")
print(f"In Colab: {app_config.in_colab}")
print(f"Local env flag: {app_config.local_env}")
print(f"GPU available: {gpu_available}")
print(f"GPU enabled by flag: {app_config.enable_gpu}")
print(f"Using GPU: {app_config.use_gpu}")

## Load and Pre-process Dataset

In [ ]:
def load_split(dataset_dir: Path, split: str) -> tuple[list[Path], np.ndarray]:
    """Load image paths and labels for train or test split."""
    image_paths: list[Path] = []
    labels: list[int] = []

    for class_name, label in LABELS.items():
        class_dir = dataset_dir / split / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing expected folder: {class_dir}")

        paths = sorted(class_dir.glob("*.png"))
        image_paths.extend(paths)
        labels.extend([label] * len(paths))

    if not image_paths:
        raise FileNotFoundError(f"No .png images found in {dataset_dir / split}")

    return image_paths, np.asarray(labels, dtype=np.int64)

# Load dataset splits
train_paths, y_train = load_split(app_config.dataset_dir, "train")
test_paths, y_test = load_split(app_config.dataset_dir, "test")

print(f"Train set: {len(train_paths)} images")
print(f"  - Good: {(y_train == 0).sum()}")
print(f"  - Not-good: {(y_train == 1).sum()}")
print(f"\nTest set: {len(test_paths)} images")
print(f"  - Good: {(y_test == 0).sum()}")
print(f"  - Not-good: {(y_test == 1).sum()}")

# Split train into train and validation
train_paths, val_paths, y_train, y_val = train_test_split(
    train_paths, y_train, test_size=app_config.validation_split, 
    random_state=42, stratify=y_train
)

print(f"\nAfter train/val split:")
print(f"Train: {len(train_paths)}, Validation: {len(val_paths)}, Test: {len(test_paths)}")

In [ ]:
# Preprocessing function selection based on model choice
def get_preprocess_fn(model_choice: str):
    if model_choice == 'VGG16':
        return vgg16_preprocess
    if model_choice == 'InceptionV3':
        return inception_preprocess
    return resnet_preprocess

preprocess_fn = get_preprocess_fn(app_config.model_choice)

print(f"Preparing tf.data datasets with {app_config.model_choice} preprocessing...")
print(f"Target size: {app_config.image_size}")

# resize and preprocess images
def decode_and_preprocess(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, app_config.image_size)
    image = tf.cast(image, tf.float32)
    image = preprocess_fn(image)
    return image, label

# Data augmentation function
# TODO validate robustness over lighting conditions
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.10)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    return image, label


def build_dataset(image_paths, labels, batch_size, is_training=False):
    image_paths = [str(path) for path in image_paths]
    labels = labels.astype(np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    dataset = dataset.map(decode_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if is_training:
        dataset = dataset.shuffle(buffer_size=min(len(image_paths), 1000), reshuffle_each_iteration=True)
        dataset = dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.cache()
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


train_dataset = build_dataset(train_paths, y_train, app_config.batch_size, is_training=True)
val_dataset = build_dataset(val_paths, y_val, app_config.batch_size)
test_dataset = build_dataset(test_paths, y_test, app_config.batch_size)

print(f"Train dataset batches: {len(train_paths) // app_config.batch_size} (+ remainder)")
print(f"Validation samples: {len(val_paths)}")
print(f"Test samples: {len(test_paths)}")


## Transfer Learning using Pre-trained Model (VGG16, InceptionV3, or ResNet50)

This cell applies a **transfer learning** approach: it loads a pre-trained CNN backbone (trained on ImageNet) and reuses its feature extractor for steel-surface classification.
 ### TODO 
- explain restnet outperformed other models
- refer to Fu J, Zhu X, Li Y (2019) for approach

### Steps:
- Create the model with ImageNet weights and without the original classifier head (`include_top=False`).
- Set the backbone to non-trainable (`base_model.trainable = False`) for initial training stability and faster convergence.

### Parameters
- `app_config.model_choice`
  - Controls which architecture is loaded: `VGG16`, `InceptionV3`, or `ResNet50`.
- `input_shape=(*app_config.image_size, 3)`
  - Defines model input size and RGB channels.
  - Must match preprocessing and resized dataset images.
- `include_top=False`
  - Removes the original ImageNet classification layers so a custom task-specific head can be added later.
- `weights='imagenet'`
  - Initializes with pre-trained ImageNet features instead of random weights.
- `base_model.trainable = False`
  - Freezes all backbone layers so only the new classification head learns during initial training.


In [ ]:
# Load pre-trained model with ImageNet weights
print(f"Loading {app_config.model_choice} with ImageNet weights...")

if app_config.model_choice == 'VGG16':
    base_model = VGG16(
        input_shape=(*app_config.image_size, 3),
        include_top=False,
        weights='imagenet' # pre-trained weights from ImageNet
    )
elif app_config.model_choice == 'InceptionV3':
    base_model = InceptionV3(
        input_shape=(*app_config.image_size, 3),
        include_top=False,
        weights='imagenet'
    )
else:  # ResNet50
    base_model = ResNet50(
        input_shape=(*app_config.image_size, 3),
        include_top=False,
        weights='imagenet'
    )

# Freeze base model layers to preserve learned features
base_model.trainable = False
print(f"Base model loaded with {len(base_model.layers)} layers")
print(f"Trainable parameters in base model: {base_model.count_params():,}")


# Classification model

This cell builds a **custom classification head** on top of the frozen pre-trained backbone (`base_model`) using a Sequential architecture.

#### Approach
- Convert spatial feature maps into a compact vector using global pooling. Lin et al., 2013, "Network In Network"
- Learn task-specific nonlinear representations with dense layers.
- Apply dropout regularization to reduce overfitting.
- Output class probability for binary classification (`not-good` as class 1).

#### Parameters
##### Architecture
- `layers.GlobalAveragePooling2D()`
  - Replaces flattening with channel-wise averaging.
  - Reduces parameter count and improves generalization.
- `layers.Dense(256, activation='relu', name='dense_1')`
  - First task-specific dense block with 256 units and ReLU activation. Nair & Hinton, 2010
- `layers.Dropout(0.5)`
  - Randomly drops 50% of units during training to reduce co-adaptation. Srivastava et al., 2014
- `layers.Dense(128, activation='relu', name='dense_2')`
  - Second dense block for refined feature learning.
- `layers.Dropout(0.3)`
  - Additional regularization with 30% dropout.
- `layers.Dense(1, activation='sigmoid', dtype='float32', name='output')`
  - Final single-unit probability output for binary classification.
  - `dtype='float32'` keeps numerically stable outputs, especially when mixed precision is enabled.
#### Training
- Adam optimizer. It scales updates differently for each weight, which helps when gradients vary across layers (common with pretrained backbones + new head).
- Early stop. Stop training if learning does not improve
- Reduce Learning rate. Reduce the learning rate when training stops improving.



In [ ]:
# Build custom classification model on top of base model
print("Building classification model...")

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    # 
    layers.Dense(256, activation='relu', name='dense_1'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu', name='dense_2'),
    layers.Dropout(0.3),
    # Output layer for binary classification
    layers.Dense(1, activation='sigmoid', dtype='float32', name='output')
])

print(f"Model architecture:")
model.summary()

# Compile the model
optimizer = keras.optimizers.Adam(learning_rate=app_config.learning_rate)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully")
print(f"Optimizer: Adam (lr={app_config.learning_rate})")
print("Loss: binary_crossentropy")
print("Metrics: accuracy")

# Define callbacks for training
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        f'steelblast_{app_config.model_choice.lower()}_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

import time

# Train the model
print(f"Training {app_config.model_choice} model...")
print(f"Epochs: {app_config.epochs}, Batch size: {app_config.batch_size}")

train_start = time.time()
history = model.fit(
    train_dataset,
    epochs=app_config.epochs,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)
train_end = time.time()
training_time_seconds = float(train_end - train_start)
print(f"\nTraining completed in {training_time_seconds:.2f} seconds!")



## 9. Evaluate Model Performance

In [ ]:
# Evaluate on test set
print("Evaluating model on test set...")
test_loss, test_accuracy = model.evaluate(test_dataset, verbose=0)
print(f"\nTest Results:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f}")

# Get predictions
# Use the model to predict on batches from the test dataset
predictions = model.predict(test_dataset, verbose=0).ravel()
y_pred = (predictions >= 0.5).astype(int)

# Extract true labels from the dataset
true_labels = np.concatenate([y for x, y in test_dataset], axis=0)

# Calculate detailed metrics
accuracy = accuracy_score(true_labels, y_pred)
precision = precision_score(true_labels, y_pred, average='weighted')
recall = recall_score(true_labels, y_pred, average='weighted')
f1 = f1_score(true_labels, y_pred, average='weighted')

print(f"\nDetailed Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

# Classification report
print(f"\nClassification Report:")
report_dict = classification_report(
    true_labels,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)
report_text = classification_report(
    true_labels,
    y_pred,
    target_names=CLASS_NAMES,
    zero_division=0
)
print(report_text)

# Confusion Matrix
cm = confusion_matrix(true_labels, y_pred)
print(f"Confusion Matrix:\n{cm}")

## 11. Save Model and Metrics

In [ ]:
# Save the trained model
model_path = f'steelblast_{app_config.model_choice.lower()}.h5'
model.save(model_path)
print(f"Model saved to {model_path}")

# Save metrics to JSON
metrics = {
    "model": app_config.model_choice,
    "image_size": app_config.image_size,
    "classification_report": report_dict,
    "dataset_info": {
        "train_samples": len(train_paths),
        "val_samples": len(val_paths),
        "test_samples": len(test_paths),
        "classes": CLASS_NAMES
    },
    "confusion_matrix": cm.tolist(),
    "training_epochs": len(history.history['loss']),
    "training_time_seconds": training_time_seconds,
    "batch_size": app_config.batch_size
}

from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
metrics_path = f"steelblast_{app_config.model_choice.lower()}_metrics_{timestamp}.json"
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=4)
print(f"Metrics saved to {metrics_path}")


# Display final summary
print("\n" + "="*60)
print(f"FINAL SUMMARY - {app_config.model_choice}")
print("="*60)
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test Loss:      {test_loss:.4f}")
print(f"Precision:      {precision:.4f}")
print(f"Recall:         {recall:.4f}")
print(f"F1-Score:       {f1:.4f}")
print("="*60)


## 12. Model Comparison and Fine-tuning Tips

### Switching Between Models
To use a different pre-trained model, modify the `model_choice` parameter in the configuration cell:
- **VGG16**: Simple, interpretable, good baseline (224×224)
- **InceptionV3**: Efficient multi-scale feature extraction (299×299)
- **ResNet50**: Better performance with residual connections (224×224)

### Fine-tuning Strategies
1. **Unfreeze Later Layers**: After initial training, unfreeze the last few layers of the base model and fine-tune with a lower learning rate
2. **Data Augmentation**: Add rotation, zoom, and horizontal flip during training for better generalization
3. **Adjust Dropout Rates**: Increase dropout if overfitting occurs
4. **Learning Rate Tuning**: Use the ReduceLROnPlateau callback to adaptively reduce learning rate

### Expected Performance
- VGG16: ~85-90% accuracy (lower complexity)
- InceptionV3: ~88-92% accuracy (balanced)
- ResNet50: ~90-93% accuracy (best overall)